# SCOTUS sanity check
Bench composition from `edge_person_sits_on_court` vs. the historical record.

In [ ]:
import duckdb, pandas as pd
DB = "graph.duckdb"          # <-- your file
con = duckdb.connect(DB, read_only=True)

def bench(d):
    return con.execute(f"""
        SELECT p.id, p.name_last, e.date_start, e.active_to, e.position_type
        FROM edge_person_sits_on_court e JOIN node_person p ON p.id = e.src_id
        WHERE e.dst_id = 'scotus' AND e.active_from <= DATE '{d}'
          AND (e.active_to IS NULL OR e.active_to > DATE '{d}')
        ORDER BY e.date_start""").df()

## 1. Bench size every year vs. statutory size
Statutory seats: 6 (1789), 7 (1807), 9 (1837), 10 (1863), attrition to 7 (1866), 9 (1869–). Vacancies mean the actual count can sit 1–2 below statute; above statute is an error.

In [ ]:
def statutory(y):
    if y < 1807: return 6
    if y < 1837: return 7
    if y < 1863: return 9
    if y < 1866: return 10
    if y < 1870: return (7, 10)      # attrition period: anything in range is fine
    return 9

rows = []
for y in range(1790, 2027):
    n = con.execute(f"""SELECT count(DISTINCT src_id) FROM edge_person_sits_on_court
                        WHERE dst_id='scotus' AND active_from <= DATE '{y}-01-01'
                          AND (active_to IS NULL OR active_to > DATE '{y}-01-01')""").fetchall()[0][0]
    s = statutory(y)
    lo, hi = (s if isinstance(s, tuple) else (s - 2, s))
    rows.append((y, n, s, "OK" if lo <= n <= hi else ("HIGH" if n > hi else "LOW")))
df = pd.DataFrame(rows, columns=["year", "sitting", "statutory", "flag"])
print(df.flag.value_counts().to_dict())
df[df.flag != "OK"]

## 2. Who is on the bench on a few dates

In [ ]:
for d in ["1803-02-24", "1857-03-06", "1954-05-17", "1973-01-22", "2020-01-01", "2024-06-01"]:
    b = bench(d)
    print(f"{d}: {len(b)} — {', '.join(b.name_last)}")

Expected on 2020-01-01: Roberts, Thomas, Ginsburg, Breyer, Alito, Sotomayor, Kagan, Gorsuch, Kavanaugh.  
On 2024-06-01: Roberts, Thomas, Alito, Sotomayor, Kagan, Gorsuch, Kavanaugh, Barrett, Jackson.

## 3. Does `active_to` match a recomputation from `raw`? (closes the scanner question)

In [ ]:
con.sql("""
SELECT count(*) AS rows,
       count(*) FILTER (WHERE e.active_to IS DISTINCT FROM
            CASE WHEN r.date_termination IS NULL THEN r.date_retirement
                 WHEN r.date_retirement IS NULL THEN r.date_termination
                 ELSE least(r.date_termination, r.date_retirement) END) AS mismatched,
       count(*) FILTER (WHERE r.date_termination IS NULL AND r.date_retirement IS NOT NULL) AS retired_no_termination
FROM edge_person_sits_on_court e JOIN raw.people_db_position r ON r.id = e.id
""").df()

`mismatched` must be 0. `retired_no_termination` is how many rows the old `least()` build got wrong.

## 4. Tenure sanity: longest and shortest SCOTUS service, and anyone with no end date who is dead

In [ ]:
con.sql("""
SELECT p.name_last, e.date_start, e.active_to, p.date_dod,
       date_diff('year', e.date_start, coalesce(e.active_to, current_date)) AS years
FROM edge_person_sits_on_court e JOIN node_person p ON p.id = e.src_id
WHERE e.dst_id = 'scotus'
ORDER BY years DESC LIMIT 5""").df()

In [ ]:
# sitting forever but dead: should be empty
con.sql("""
SELECT p.name_last, e.date_start, p.date_dod, e.position_type
FROM edge_person_sits_on_court e JOIN node_person p ON p.id = e.src_id
WHERE e.dst_id = 'scotus' AND e.active_to IS NULL AND p.date_dod IS NOT NULL""").df()

## 5. Same check for a circuit: active Ninth Circuit judges by decade
Statutory active judgeships: 29 since 1984 (28 from 1978). Senior judges should be excluded.

In [ ]:
for y in range(1950, 2030, 10):
    n = con.execute(f"""SELECT count(DISTINCT src_id) FROM edge_person_sits_on_court
                        WHERE dst_id='ca9' AND active_from <= DATE '{y}-01-01'
                          AND (active_to IS NULL OR active_to > DATE '{y}-01-01')""").fetchall()[0][0]
    print(y, n)